# Dataset

> Base class of episode datasets: clips of `num_steps` steps for training (stable-worldmodel API).

In [ ]:
#| default_exp data.dataset

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import os
from pathlib import Path
from typing import Callable

import numpy as np

In [ ]:
#| export
def get_cache_dir(cache_dir: str | Path | None = None, sub_folder: str | None = None) -> Path:
    "Where named datasets live: `cache_dir`, else ``$STABLEMARL_HOME``, else ``~/.stable_marl``."
    root = Path(cache_dir or os.environ.get('STABLEMARL_HOME', '~/.stable_marl')).expanduser()
    path = root / sub_folder if sub_folder else root
    path.mkdir(parents=True, exist_ok=True)
    return path


def to_tensor(data: np.ndarray):
    """
    A column slice as a torch tensor. As in stable-worldmodel, images of shape (T, H, W, C)
    with 1 or 3 channels become channel-first, (T, C, H, W); per-agent images
    (T, num_agents, H, W, C) are left as they are.
    """
    import torch
    tensor = torch.from_numpy(np.ascontiguousarray(data))
    if tensor.ndim == 4 and tensor.shape[-1] in (1, 3):
        tensor = tensor.permute(0, 3, 1, 2)
    return tensor

In [ ]:
#| export
class Dataset:
    """
    Base class of episode datasets, stored as flat columns with episode ``lengths`` / ``offsets``.

    Item ``i`` is a clip of ``num_steps`` steps, ``frameskip`` apart: a dict of torch tensors,
    one per column. The ``action`` column keeps every step of the clip and is reshaped to
    ``(num_steps, -1)``, so with ``frameskip=1`` it is ``(num_steps, num_agents)``.
    Subclasses implement ``column_names`` and ``_load_slice``.
    """
    def __init__(
        self,
        lengths: np.ndarray,
        offsets: np.ndarray,
        frameskip: int = 1,
        num_steps: int = 1,
        transform: Callable[[dict], dict] | None = None):
        self.lengths, self.offsets = lengths, offsets
        self.frameskip, self.num_steps = frameskip, num_steps
        self.span = num_steps * frameskip
        self.transform = transform
        self.clip_indices = [(ep, start) for ep, length in enumerate(lengths) if length >= self.span
                             for start in range(length - self.span + 1)]

    @property
    def column_names(self) -> list[str]:
        raise NotImplementedError

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        raise NotImplementedError

    def __len__(self) -> int:
        return len(self.clip_indices)

    def __getitem__(self, idx: int) -> dict:
        ep_idx, start = self.clip_indices[idx]
        steps = self._load_slice(ep_idx, start, start + self.span)
        if 'action' in steps:
            steps['action'] = steps['action'].reshape(self.num_steps, -1)
        return steps

    def load_chunk(self, episodes_idx: np.ndarray, start: np.ndarray, end: np.ndarray) -> list[dict]:
        "One slice ``[start, end)`` per episode, in order."
        chunk = []
        for ep, s, e in zip(episodes_idx, start, end):
            steps = self._load_slice(ep, s, e)
            if 'action' in steps:
                steps['action'] = steps['action'].reshape((e - s) // self.frameskip, -1)
            chunk.append(steps)
        return chunk

    def load_episode(self, episode_idx: int) -> dict:
        "Every step of one episode."
        return self._load_slice(episode_idx, 0, self.lengths[episode_idx])

    def get_col_data(self, col: str) -> np.ndarray:
        "The whole column, rows of every episode one after the other."
        raise NotImplementedError

    def get_dim(self, col: str) -> int:
        "Number of values per row of `col`."
        data = self.get_col_data(col)
        return int(np.prod(data.shape[1:])) if data.ndim > 1 else 1

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()